# 00 – Carga da base

Issue #2. Carrega a base acadêmica v7 com `carregar_base()` e mostra os totais de referência do projeto. Todos os outros notebooks partem desta mesma função.

Antes de rodar, defina a variável de ambiente `BASE_CSV_PATH` com o caminho do `Base_academica.csv` (veja a seção **Dados** do README).

> Este notebook mostra só totais e tipos. Nenhuma linha da base e nenhum `HANDLE*` aparece nas saídas.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Permite importar src/ rodando o notebook a partir da pasta notebooks/ ou da raiz.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.dados import carregar_base

pd.set_option("display.max_rows", 100)

In [2]:
df = carregar_base()
print("Base carregada.")

Base carregada.


## Totais de referência

Comparação com a ficha da `Legenda.md` (seções 1 e 3). A contagem de colunas desconsidera as duas colunas criadas pela função (`ANOSEMESTRE_ORD` e `INGRESSO_ORD`).

In [3]:
COLUNAS_CRIADAS = ["ANOSEMESTRE_ORD", "INGRESSO_ORD"]

esperado_legenda = {
    "Linhas": 59_059,
    "Colunas do arquivo": 66,
    "Alunos distintos": 7_615,
    "Semestres": 35,
    "Códigos de curso": 12,
    "Linhas ACADEMICA": 44_417,
    "Linhas OCORRENCIA": 14_642,
}

origem = df["ORIGEMREGISTRO"].value_counts()
obtido = {
    "Linhas": len(df),
    "Colunas do arquivo": df.shape[1] - len(COLUNAS_CRIADAS),
    "Alunos distintos": df["HANDLEALUNO"].nunique(),
    "Semestres": df["ANOSEMESTRE"].nunique(),
    "Códigos de curso": df["SIGLACURSO"].nunique(),
    "Linhas ACADEMICA": int(origem.get("ACADEMICA", 0)),
    "Linhas OCORRENCIA": int(origem.get("OCORRENCIA", 0)),
}

totais = pd.DataFrame({"Legenda v7": esperado_legenda, "Obtido": obtido})
totais["Confere"] = totais["Legenda v7"] == totais["Obtido"]
totais

,Legenda v7,Obtido,Confere
Linhas,59059,59059,True
Colunas do arquivo,66,66,True
Alunos distintos,7615,7615,True
Semestres,35,35,True
Códigos de curso,12,12,True
Linhas ACADEMICA,44417,44417,True
Linhas OCORRENCIA,14642,14642,True


In [4]:
ordem = df["ANOSEMESTRE_ORD"]
primeiro = df.loc[ordem.idxmin(), "ANOSEMESTRE"]
ultimo = df.loc[ordem.idxmax(), "ANOSEMESTRE"]
intervalo = int(ordem.max() - ordem.min() + 1)

print(f"Primeiro semestre: {primeiro}")
print(f"Último semestre:   {ultimo}")
print(f"Semestres com dados: {df['ANOSEMESTRE'].nunique()} de {intervalo} no intervalo")

Primeiro semestre: 2009/1
Último semestre:   2026/1
Semestres com dados: 35 de 35 no intervalo


## Códigos de curso

A legenda descreve 12 códigos (8 engenharias, 3 cursos superiores de tecnologia e 1 código de aluno especial), mas não lista quais são. A identificação de cada código é tarefa da issue #3.

In [5]:
sorted(df["SIGLACURSO"].unique())

['GAE',
 'GEA',
 'GEB',
 'GEC',
 'GEE',
 'GEL',
 'GEP',
 'GES',
 'GET',
 'TAI',
 'TGT',
 'TRC']

## Linhas por origem do registro

In [6]:
pd.DataFrame({
    "linhas": origem,
    "%": (origem / len(df) * 100).round(1),
})

,linhas,%
ORIGEMREGISTRO,,
ACADEMICA,44417,75.2
OCORRENCIA,14642,24.8


## Colunas e tipos

Tipos atribuídos na leitura: identificadores `HANDLE*` como inteiro que aceita vazio (`Int64`), colunas `DATA*` como data, números com vírgula decimal como `float64`. Colunas de contagem com algum vazio aparecem como `float64`, porque o vazio não cabe em `int64`.

In [7]:
pd.DataFrame({
    "posição": range(1, df.shape[1] + 1),
    "tipo": df.dtypes.astype(str),
})

,posição,tipo
HANDLEANOSEMESTRE,1,Int64
ANOSEMESTRE,2,str
HANDLEALUNOANOSEMESTRE,3,Int64
HANDLEALUNO,4,Int64
HANDLEALUNOCURRICULO,5,Int64
HANDLECURRICULO,6,Int64
PERIODO,7,float64
SIGLACURSO,8,str
TURNO,9,str
CRENOSEMESTRE,10,float64
